# Archiving Workflow

Archiving magnetotelluric data typically follows a three-step process: creating a holding site wherever the data is to be archived (such as Science Base), loading the data along with its metadata and transfer functions, and finally exporting it to the archiving site as an XML. This notebook will aim to thoroughly explain each of these steps and provide relevant code examples for each substep.

# 1. USGS ScienceBase Data Release Process  
Releasing data through **ScienceBase** is how USGS makes scientific datasets publicly available in a standardized, citable, and reviewable way. The process ensures that data are well‑documented, approved, and permanent. Here is a general overview on how data is typically released on ScienceBase.

---

## 1. Plan Your Data Release Early  
Before you begin collecting or analyzing data, identify that you will need to release it.

Early planning helps you determine:  
- What data will be released  
- Who is responsible for review  
- What supplemental files (metadata, readme, code) will be required

---

## 2. Prepare Your Data  
Create a complete, well‑organized package that anyone could understand.

This typically includes:  
- Clean final datasets  
- Methods descriptions  
- Documentation explaining how to read/use the files  
- Optional reproducibility scripts

---

## 3. Write the Metadata  
Metadata describes the who, what, when, where, why, and how of the dataset.

Good metadata covers:  
- Purpose of the data  
- Collection and processing methods  
- Explanation of each field  
- Use constraints  
- Contact information

USGS uses standards like **FGDC** or **ISO**. Tools such as the **USGS Metadata Wizard** can help.

---

## 4. Complete Scientific Review  
A qualified reviewer checks your data and metadata for:  
- Accuracy  
- Completeness  
- Clarity  
- Policy compliance

---

## 5. Complete Bureau Approval  
USGS requires official approval to ensure compliance with **Fundamental Science Practices (FSP)**.

This step may involve:  
- Submitting materials through internal review systems  
- Getting sign‑off from reviewers and supervisors  
- Ensuring related publications are linked

---

## 6. Publish the Data Release in ScienceBase  
A ScienceBase librarian or data manager will help finalize the public landing page.

You will provide:  
- Final data files  
- Final metadata  
- Abstract and description  
- Keywords and contacts

Once published, ScienceBase assigns a **DOI** and hosts your data permanently.

---

## 7. Announce & Cite the Data Release  
After publication:  
- Share the DOI  
- Use the official citation in your manuscripts  
- Notify collaborators

# 2. Preparing Data For Archival

MTH5 is both a compact self-descriptive data format that can house both MT data and associated metadata, and the name of USGS's in-house Python library dedicated to creating files in that format. It contains multiple readers for various file formats including NIMS, Phoenix, and Z3D, and most of them can be invoked rather simply with a few lines of code. For example, a NIMS .bin file can be imported and read via something like this:

In [ ]:
from mth5.mth5 import MTH5
from mth5.io.nims import NIMSCollection
from mth5 import read_file
from mth5.clients import MakeMTH5

mth5_path = MakeMTH5.from_nims("path/to/nims/data.bin")

In the not unlikely scenario your data involves the use of transfer functions, you will need to create those transfer functions, along with their associated images and metadata, and compress them into a ZIP file. mtpy_v2 has ways to accomplish this. To read in and apply transfer functions, navigate to where those files are located and then you can run something such as:

In [ ]:
from mtpy import MT, MTCollection 

for fn in PROFILE_LIST:
    mt_object = MT()
    mt_object.read(fn)
    mt_object.survey_metadata.id = "profile"
    mtc.add_tf(mt_object)

for fn in GRID_LIST:
    mt_object = MT()
    mt_object.read(fn)
    mt_object.survey_metadata.id = "grid"
    mtc.add_tf(mt_object)

In the example, we also supply survey metadata in case there are known to be none present ahead of time. Then the transfer functions can be plotted via matplotlib or your favorite graphical package and stored in a compressed folder.

Metadata is housed in an XML file. You can find an example XML file [here](https://github.com/jpopelar/mt_sb_archiving/blob/main/xml_templates/mt_child_template.xml) that you can fill in with whatever appropriate metadata your upload requires.

# 3. Uploading

Once everything has been gathered and properly formatted, it is time to upload the data to ScienceBase. The sciencebasepy library holds all the tools that we will need in order to make this happen. Since this process is a little more involved, we will include some proper example code that you can run directly, to eliminate the chances of something going wrong.

In [ ]:
import sciencebasepy as sb
import time
from pathlib import Path

Your data should have been assigned a parent page to upload to. Provide the parent ID - the hex code at the end of the ScienceBase web address - in the cell below:

In [ ]:
parent_id = "0123456789abcdef"

To get a ScienceBase session started, we'll need an API token to log in with. Start the session first via...

In [ ]:
session = sb.Session()
session.get_token()

You should then be able to copy the API token as a full dictionary, paste it into the cell below and log in. If the following cell prints True for Is Logged In, then everything went to plan:

In [ ]:
token = {
    "access_token": "eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJCMWJfZFBNR3BWY09hT253dU0zQThnSGF2Mzd5U1Y0OXJaLUZ2U2gtaXRNIn0.eyJleHAiOjE3NDQzMjIzODYsImlhdCI6MTc0NDMyMDU4NiwiYXV0aF90aW1lIjoxNzQ0MzIwMjM0LCJqdGkiOiIxNWJhOTY3Zi04ZWEwLTRkYTAtOTFkOC1lMGE0Nzk0NzZlYTEiLCJpc3MiOiJodHRwczovL3d3dy5zY2llbmNlYmFzZS5nb3YvYXV0aC9yZWFsbXMvU2NpZW5jZUJhc2UiLCJhdWQiOiJhY2NvdW50Iiwic3ViIjoiOTY1OWVkYmQtOWU0ZC00MWE0LWFhNDEtNWU0YTM3ODM2YTZlIiwidHlwIjoiQmVhcmVyIiwiYXpwIjoiY2F0YWxvZyIsIm5vbmNlIjoiNTVjMTAwYWYtZGY0ZC00NDkyLThlNDYtM2Y5NTNiNzE5NDM5Iiwic2Vzc2lvbl9zdGF0ZSI6IjNkYWNkYzU3LWU1YjgtNGQ1Yy04ZWY3LTc3Y2E3NWJiODA0NiIsImFjciI6IjAiLCJhbGxvd2VkLW9yaWdpbnMiOlsiaHR0cHM6Ly93d3cuc2NpZW5jZWJhc2UuZ292L3VwbG9hZCIsImh0dHBzOi8vYXBpLnNjaWVuY2ViYXNlLmdvdi8qIiwiKiIsImh0dHBzOi8vYmV0YS5zY2llbmNlYmFzZS5nb3YvZGlyZWN0b3J5IiwiaHR0cDovL2xvY2FsaG9zdDozMDAwIiwiaHR0cDovL2xvY2FsaG9zdDo0MDAwLyoiLCJodHRwczovL2Rldi1zY2llbmNlYmFzZS51c2dzLmdvdiIsImh0dHBzOi8vYmV0YS5zY2llbmNlYmFzZS5nb3YiLCJodHRwOi8vbG9jYWxob3N0OjgwODAiLCJodHRwczovL3NjaWVuY2ViYXNlLmdvdiIsImh0dHBzOi8vd3d3LnNjaWVuY2ViYXNlLmdvdi9jYXRhbG9nIiwiaHR0cDovL2xvY2FsaG9zdDo4MDkwIiwiaHR0cDovL2xvY2FsaG9zdDo4MDg4LyoiLCJodHRwczovL2JldGEuc2NpZW5jZWJhc2UuZ292L3VwbG9hZCIsImh0dHBzOi8vYmV0YS5zdGFnaW5nLnNjaWVuY2ViYXNlLmdvdi8qIiwiaHR0cHM6Ly9iZXRhLnNjaWVuY2ViYXNlLmdvdi92b2NhYiJdLCJyZWFsbV9hY2Nlc3MiOnsicm9sZXMiOlsib2ZmbGluZV9hY2Nlc3MiLCJ1bWFfYXV0aG9yaXphdGlvbiJdfSwicmVzb3VyY2VfYWNjZXNzIjp7ImFjY291bnQiOnsicm9sZXMiOlsidmlldy1wcm9maWxlIl19fSwic2NvcGUiOiJvcGVuaWQgcHJvZmlsZSBlbWFpbCIsImVtYWlsX3ZlcmlmaWVkIjp0cnVlLCJuYW1lIjoiSmFyZWQgUGVhY29jayIsInByZWZlcnJlZF91c2VybmFtZSI6ImpwZWFjb2NrQHVzZ3MuZ292IiwiZ2l2ZW5fbmFtZSI6IkphcmVkIiwiZmFtaWx5X25hbWUiOiJQZWFjb2NrIiwiZW1haWwiOiJqcGVhY29ja0B1c2dzLmdvdiJ9.lCpNd3JSGmfpc5hN0nBqW05kkVCN2yo1q3XOi9Q2OhxdLC7l_BNRLUCLfseHtWAzaRkP7JwqvV2jLysAN6n3N-mdxLCnXBCvbXRpc8Fd5VD40ZNCvCM1JFCWH6iymlaOxdF5VYKZ2IUuABr_WJsuJ4vbGSPxa-iazDSo5S5oBhLGfI3JmyvF2CyG2glkzt5gg188h8cnozWWv5TmV5UR9Mhuyk0_W6WMR8n7YCq2se5wGCOVzBJP5DrQiZdnnxnSQi_t4lE8rkco9KGNkR-zcX_Hq0GZ0lx4t7a9VZCFhqbJGLlrCDZmSQWiUvQ83Yqat-Ub6k4Ew4DPkYX0FJ01EA",
    "refresh_token": "eyJhbGciOiJIUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJkMmMyMmZmMy1mZjIxLTQ4NjQtYjJlMS04NTQ1ZWJlNmU0NjkifQ.eyJleHAiOjE3NDQzMjQ1NDYsImlhdCI6MTc0NDMyMDU4NiwianRpIjoiODA5NWM0NTktNzgyZS00MzUwLTliMmEtNWUzYzE2MjQzZDBiIiwiaXNzIjoiaHR0cHM6Ly93d3cuc2NpZW5jZWJhc2UuZ292L2F1dGgvcmVhbG1zL1NjaWVuY2VCYXNlIiwiYXVkIjoiaHR0cHM6Ly93d3cuc2NpZW5jZWJhc2UuZ292L2F1dGgvcmVhbG1zL1NjaWVuY2VCYXNlIiwic3ViIjoiOTY1OWVkYmQtOWU0ZC00MWE0LWFhNDEtNWU0YTM3ODM2YTZlIiwidHlwIjoiUmVmcmVzaCIsImF6cCI6ImNhdGFsb2ciLCJub25jZSI6IjU1YzEwMGFmLWRmNGQtNDQ5Mi04ZTQ2LTNmOTUzYjcxOTQzOSIsInNlc3Npb25fc3RhdGUiOiIzZGFjZGM1Ny1lNWI4LTRkNWMtOGVmNy03N2NhNzViYjgwNDYiLCJzY29wZSI6Im9wZW5pZCBwcm9maWxlIGVtYWlsIn0.b0pU_pxg5MWqCvXveAMEMGJTRyJG7g1Gve6IG3SLK2k",
}
session.add_token(token)
print(f"Is Logged In: {session.is_logged_in()}")

Now for the actual uploading process. What we'll do is get a JSON file for the parent page and add onto it with our own files. Getting that JSON is as simple as executing...

In [ ]:
item = session.get_item(parent_id)

Let us also specify the directory where all of our files - H5, XML, ZIP, and whatever else - are residing...

In [ ]:
h5_path = Path(r"C:\\path\\to\\my\\data\\folder")

We'll ultimately want to upload everything in that folder to the JSON but for the sake of brevity and simplicity, let us focus on the H5 files for now. This loop will scan over all of the .h5 files in our data directory and upload any new ones to the parent page. Since the connection to ScienceBase can be a little finicky at times, we will also check for any failed or timed out uploads and possibly run this cell again to cover them off.

In [ ]:
for h5_file in list(h5_path.glob("*.h5")):
    if h5_file.name in [d["name"] for d in item["files"]]:
        print(f"Skipping: {h5_file.name} is already uploaded")
        continue
    try:
        item = session.upload_file_to_item(item, h5_file.as_posix(), scrape_file=False)
        print(f"{time.ctime()} Uploaded: {h5_file.name}")
    except Exception as error:
        print(f"{h5_file.name} Error: {error}")
        print(f"Is Logged In: {session.is_logged_in()}")
        item = session.get_item(parent_id)
        continue

fns = [d["name"] for d in item["files"]] 
for a in h5_path.glob("*.h5"):
    if a.name not in fns:
        print(a.name)

You can upload your images and metadata XML via similar calls to ``session.upload_file_to_item()``. One last thing we may want to do at the end is reorganize our uploads; since the JSON holds them on a first come first serve basis, we can get it to alphabetize the files instead for readability. This sounds intensive but really we just need to sort our own list and then override the one on the page. 

These next few cells will produce a **lot** of output. You have been warned.

In [ ]:
initial_files = []
h5_files = []
for fn in fns:
    if fn.endswith(".xml"):
        initial_files.append(fn)
    elif fn.endswith(".zip"):
        initial_files.append(fn)
    else:
        h5_files.append(fn)

ordered_fns = initial_files + sorted(h5_files)
for fn in ordered_fns:
    print(fn)

In [ ]:
new_file_list = []

for fn in ordered_fns:
    for file_item in item["files"]:
        if file_item["name"] == fn:
            new_file_list.append(file_item)
            break

item["files"] = new_file_list
for d in item["files"]:
    print(d["name"])

In [ ]:
session.update_item({"id": parent_id, "files": new_file_list})

And with our upload complete the last thing to do is simply close our session and celebrate our contribution to science!

In [ ]:
session.logout()